In [1]:

# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import soundfile as sf
import pandas as pd

print("✅ Imports done")

✅ Imports done


In [2]:

# ============================================================
# CELL 2 — Paths and thresholds
# ============================================================
BASE_DIR      = "/Users/abey/Documents/duration_ratio"   # ← change this
REFERENCE_DIR = os.path.join(BASE_DIR, "reference")
MODELS_DIR    = os.path.join(BASE_DIR, "models")

DURATION_TOLERANCE = 0.10   # ±10% — calibrate with editor later

print(f"Tolerance : ±{int(DURATION_TOLERANCE * 100)}%  (ratio must be {1 - DURATION_TOLERANCE:.2f} – {1 + DURATION_TOLERANCE:.2f})")
print("✅ Paths and thresholds set")



Tolerance : ±10%  (ratio must be 0.90 – 1.10)
✅ Paths and thresholds set


In [3]:

# ============================================================
# CELL 3 — get_duration function
# ============================================================
def get_duration(file_path):
    """Fast duration extraction — reads WAV header only, no decode."""
    with sf.SoundFile(file_path) as f:
        return f.frames / f.samplerate

print("✅ get_duration defined")



✅ get_duration defined


In [4]:

# ============================================================
# CELL 4 — Startup validation
# ============================================================
for folder in [BASE_DIR, REFERENCE_DIR, MODELS_DIR]:
    if not os.path.exists(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")
print("✅ Top level folders found")

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([f for f in os.listdir(model_path) if f.endswith(".wav")])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

# cross-model filename validation
reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
for wav_file in sample_names:
    ref_path = os.path.join(REFERENCE_DIR, wav_file)
    if not os.path.exists(ref_path):
        raise FileNotFoundError(f"Missing reference for {wav_file} — expected: {ref_path}")
print("✅ All reference files found")

total = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")




✅ Top level folders found
✅ Models found: ['m1']
   m1: 2 samples
✅ All models have identical filenames
✅ All reference files found

Ready: 1 models × 2 samples = 2 evaluations


In [5]:

# ============================================================
# CELL 5 — Main evaluation loop
# ============================================================
results = []

lower_bound = 1.0 - DURATION_TOLERANCE
upper_bound = 1.0 + DURATION_TOLERANCE

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        tts_path    = os.path.join(MODELS_DIR, model, wav_file)
        ref_path    = os.path.join(REFERENCE_DIR, wav_file)

        print(f"\n  Sample : {sample_name}")

        try:
            ref_dur = get_duration(ref_path)
            tts_dur = get_duration(tts_path)
            ratio   = round(tts_dur / ref_dur, 4) if ref_dur > 0 else None

            if ratio is None:
                final_pass   = "⚠️ ERROR"
                failure_type = "Zero ref duration"
            elif ratio < lower_bound:
                final_pass   = "❌ FAIL"
                failure_type = "Too Short"
            elif ratio > upper_bound:
                final_pass   = "❌ FAIL"
                failure_type = "Too Long"
            else:
                final_pass   = "✅ PASS"
                failure_type = "—"

            print(f"  Ref    : {round(ref_dur, 3)}s | TTS: {round(tts_dur, 3)}s | Ratio: {ratio} → {final_pass}")

            results.append({
                "Model"       : model,
                "Sample"      : sample_name,
                "Ref Dur"     : round(ref_dur, 3),
                "TTS Dur"     : round(tts_dur, 3),
                "Ratio"       : ratio,
                "Final Pass"  : final_pass,
                "Failure Type": failure_type,
            })

        except Exception as e:
            print(f"  🚨 ERROR: {e}")
            results.append({
                "Model"       : model,
                "Sample"      : sample_name,
                "Ref Dur"     : None,
                "TTS Dur"     : None,
                "Ratio"       : None,
                "Final Pass"  : "⚠️ ERROR",
                "Failure Type": str(e),
            })

print("\n\nAll evaluations complete.")




Model: m1

  Sample : YASH 2
  Ref    : 332.435s | TTS: 254.855s | Ratio: 0.7666 → ❌ FAIL

  Sample : YASH_01
  Ref    : 254.855s | TTS: 332.435s | Ratio: 1.3044 → ❌ FAIL


All evaluations complete.


In [6]:

# ============================================================
# CELL 6 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

# ── Table 1 — full per segment results ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[[
    "Model", "Sample", "Ref Dur", "TTS Dur", "Ratio", "Final Pass", "Failure Type"
]].to_string(index=False))

# ── Table 2 — per model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df     = df[df["Model"] == model]
    total        = len(model_df)
    pass_count   = (model_df["Final Pass"] == "✅ PASS").sum()
    short_count  = (model_df["Failure Type"] == "Too Short").sum()
    long_count   = (model_df["Failure Type"] == "Too Long").sum()
    error_count  = (model_df["Final Pass"] == "⚠️ ERROR").sum()
    valid_ratios = model_df["Ratio"].dropna()

    summary_rows.append({
        "Model"        : model,
        "Segments"     : total,
        "Pass Rate"    : f"{pass_count}/{total}",
        "Too Short"    : short_count,
        "Too Long"     : long_count,
        "Errors"       : error_count,
        "Median Ratio" : round(valid_ratios.median(), 4) if len(valid_ratios) > 0 else None,
        "Mean Ratio"   : round(valid_ratios.mean(), 4)   if len(valid_ratios) > 0 else None,
        "Min Ratio"    : round(valid_ratios.min(), 4)    if len(valid_ratios) > 0 else None,
        "Max Ratio"    : round(valid_ratios.max(), 4)    if len(valid_ratios) > 0 else None,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")
print("Primary   → Pass Rate (highest first)")
print("Tiebreak1 → Median Ratio closest to 1.0 (least abs deviation)")
print("Tiebreak2 → Too Short count (lowest first — short is worse than long for dialogue)\n")

summary_df["_pass_num"]      = summary_df["Pass Rate"].apply(lambda x: int(x.split("/")[0]))
summary_df["_ratio_dev"]     = summary_df["Median Ratio"].apply(
    lambda x: abs(x - 1.0) if x is not None else 999
)

ranking = summary_df.sort_values(
    by=["_pass_num", "_ratio_dev", "Too Short"],
    ascending=[False, True, True]
)[[
    "Model", "Pass Rate", "Median Ratio", "Mean Ratio",
    "Too Short", "Too Long", "Errors"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Pass Rate    → % of segments within ±10% duration tolerance")
print("Median Ratio → 1.0 = perfect match | <0.9 too short | >1.1 too long")
print("Too Short    → dialogue cut off — worse than too long for dubbing")
print("Too Long     → model padding or hallucinating silence")
print("Errors       → corrupt or unreadable files — check manually")
print(f"\nTolerance: ±{int(DURATION_TOLERANCE * 100)}% (ratio {lower_bound:.2f} – {upper_bound:.2f})")



========== FULL PER-SEGMENT RESULTS ==========
Model  Sample  Ref Dur  TTS Dur  Ratio Final Pass Failure Type
   m1  YASH 2  332.435  254.855 0.7666     ❌ FAIL    Too Short
   m1 YASH_01  254.855  332.435 1.3044     ❌ FAIL     Too Long

========== MODEL COMPARISON SUMMARY ==========
Model  Segments Pass Rate  Too Short  Too Long  Errors  Median Ratio  Mean Ratio  Min Ratio  Max Ratio
   m1         2       0/2          1         1       0        1.0355      1.0355     0.7666     1.3044

========== MODEL RANKING ==========
Primary   → Pass Rate (highest first)
Tiebreak1 → Median Ratio closest to 1.0 (least abs deviation)
Tiebreak2 → Too Short count (lowest first — short is worse than long for dialogue)

Model Pass Rate  Median Ratio  Mean Ratio  Too Short  Too Long  Errors
   m1       0/2        1.0355      1.0355          1         1       0

========== WHAT TO LOOK FOR ==========
Pass Rate    → % of segments within ±10% duration tolerance
Median Ratio → 1.0 = perfect match | <0.9 too 

In [7]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
